In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
data = pd.read_excel("ENB2012_data.xlsx")
data.head()
data.shape
data.isnull().sum()
# comment about dataset: The dataset represents different building geometries and their corresponding heating and cooling
# energy requirements. The objective is to develop a data-driven model to predict energy demand and identify important
# design parameters.






FileNotFoundError: [Errno 2] No such file or directory: 'ENB2012_data.xlsx'

In [ ]:
#Are most buildings low or high energy demand?
plt.hist(data["Y1"])
plt.xlabel("Heating Load")
plt.ylabel("Frequency")

In [ ]:
#Correlation Analysis between variables
plt.figure(figsize=(10,8))
sns.heatmap(data.corr(),annot=True)
plt.show()

In [ ]:
#Engineering Relationship Plot
#Surface area vs heating load
plt.scatter(data["X2"],data["Y1"])
plt.xlabel("Surface Area")
plt.ylabel("Heating Load")
plt.show()


In [ ]:
#Separate Input and Output
X=data.iloc[:,0:8]
y=data["Y1"]
#Split Data
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


In [ ]:
#Model 1: Linear Regression
from sklearn.linear_model import LinearRegression
lr=LinearRegression()
lr.fit(X_train,y_train)
lr_prediction=lr.predict(X_test)


In [ ]:
#Model 2: Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor
rf=RandomForestRegressor(n_estimators=100,random_state=42)
rf.fit(X_train,y_train)
rf_prediction=rf.predict(X_test)


In [ ]:
#Model 3: Gradient Boosting Regressor
from sklearn.ensemble import GradientBoostingRegressor
gb=GradientBoostingRegressor()
gb.fit(X_train,y_train)
gb_prediction=gb.predict(X_test)


In [ ]:
#Model 4: Neural Network
from sklearn.neural_network import MLPRegressor
ann=MLPRegressor(hidden_layer_sizes=(50,50),max_iter=1000)
ann.fit(X_train,y_train)


In [ ]:
#Evaluate Models
from sklearn.metrics import r2_score,mean_squared_error
print("Random Forest:",r2_score(y_test,rf_prediction))


In [ ]:
from sklearn.metrics import r2_score,mean_squared_error
import pandas as pd

#create table
lr_prediction = lr.predict(X_test)
rf_prediction = rf.predict(X_test)
gb_prediction = gb.predict(X_test)
ann_prediction = ann.predict(X_test)

results = { "Model": [ "Linear Regression","Random Forest", "Gradient Boosting",
                      "ANN"],
  "R² Score": [
        r2_score(y_test, lr_prediction),
        r2_score(y_test, rf_prediction),
        r2_score(y_test, gb_prediction),
        r2_score(y_test, ann_prediction)
    ],

    "RMSE": [
        mean_squared_error(y_test, lr_prediction)**0.5,
        mean_squared_error(y_test, rf_prediction)**0.5,
        mean_squared_error(y_test, gb_prediction)**0.5,
        mean_squared_error(y_test, ann_prediction)**0.5
    ]
}

comparison_table = pd.DataFrame(results)
comparison_table

comparison_table.sort_values(by="R² Score", ascending=False)


In [ ]:
#Add Explainable AI (SHAP)
!pip install shap
import shap
explainer=shap.TreeExplainer(rf)
shap_values=explainer.shap_values(X_test)
shap.summary_plot(shap_values,X_test)



In [ ]:
#Optimization
results=[]
for glazing in np.linspace(0.1,0.5,20):
    sample=X_test.iloc[0].copy()
    sample["X7"]=glazing
    energy=rf.predict([sample] )
    results.append(energy[0] )
plt.plot(results)
plt.xlabel("Glazing Variation")
plt.ylabel("Predicted Heating Load")
plt.show()